# SLC46A1 engraftment, step 2: exhaustive rescue scan and gate perturbation probe

Scores every single substitution that can be made anywhere in chicken PCFT and reads two
things off each forward pass.

For each high-risk pharmacophore position, the probability ESM-2 assigns to the human
residue there. A substitution that lifts it back above 1e-3 is a rescuer.

For each of the five extracellular gate positions of Zhao and colleagues, the probability
assigned to the chicken wild-type residue there. Substituting at one member of a measured
cross-linking pair and reading its partner gives a comparison against measured coupling
rather than against another prediction.

One pass serves every read-out position, so the scan covers the whole protein instead of a
fixed flanking window, and both the window radius and the treatment of identical-target
positions become filters applied afterwards. Progress is checkpointed to Drive. Every cell
aborts on a failed precondition.

In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step2')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/pharmacophore_position_set.tsv',
            'data/primary_substitution_risk.tsv',
            'data/zhao_gate_chicken.json',
            'structures/7BC6_to_human.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))

CK2HU = {int(k): v for k, v in json.load(
    open(os.path.join(PROJECT, 'structures', '7BC6_to_human.json')))['ck2hu'].items()}
log('sequences ok: human %d, chicken %d' % (len(human), len(chicken)))


In [ ]:
THRESHOLD = 1e-3

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
col = {k: i for i, k in enumerate(rows[0])}

targets = []
for r in rows[1:]:
    if r[col['high_risk']] != 'True':
        continue
    cp = int(r[col['chicken_pos']])
    if chicken[cp - 1] != r[col['chicken_aa']]:
        raise SystemExit('FATAL chicken %d is %s in sequence, %s in table'
                         % (cp, chicken[cp - 1], r[col['chicken_aa']]))
    targets.append({'chicken_pos': cp, 'chicken_aa': r[col['chicken_aa']],
                    'human_pos': int(r[col['human_pos']]),
                    'human_aa': r[col['human_aa']],
                    'p_baseline': float(r[col['p_human_wt_background']])})
if not targets:
    raise SystemExit('FATAL no high-risk positions in the step 1 results')
for t in targets:
    if t['p_baseline'] >= THRESHOLD:
        raise SystemExit('FATAL %s%d baseline %.3e is not below the threshold'
                         % (t['chicken_aa'], t['chicken_pos'], t['p_baseline']))

gates = json.load(open(os.path.join(PROJECT, 'data', 'zhao_gate_chicken.json')))
for g in gates:
    if chicken[g['chicken_pos'] - 1] != g['chicken_aa']:
        raise SystemExit('FATAL gate chicken %d is %s in sequence, %s in file'
                         % (g['chicken_pos'], chicken[g['chicken_pos'] - 1],
                            g['chicken_aa']))

log('rescue targets %s' % ', '.join('%s%d->%s' % (t['chicken_aa'], t['chicken_pos'],
                                                  t['human_aa']) for t in targets))
log('gate probes %s' % ', '.join('%s%d' % (g['chicken_aa'], g['chicken_pos'])
                                 for g in gates))


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
AA = 'ACDEFGHIKLMNPQRSTVWY'
SCHEMA = 'v2-primary-and-gate'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()

base_ids = tok(chicken, return_tensors='pt')['input_ids'][0]
offset = 1

read_idx = ([t['chicken_pos'] - 1 + offset for t in targets]
            + [g['chicken_pos'] - 1 + offset for g in gates])
read_tok = ([tok.convert_tokens_to_ids(t['human_aa']) for t in targets]
            + [tok.convert_tokens_to_ids(g['chicken_aa']) for g in gates])
cols = (['p_%d' % t['chicken_pos'] for t in targets]
        + ['g_%d' % g['chicken_pos'] for g in gates])

variants = [(j, y) for j in range(1, len(chicken) + 1)
            for y in AA if y != chicken[j - 1]]
log('variants to score %d, read-out positions %d' % (len(variants), len(read_idx)))

CKPT = os.path.join(OUT, 'scan_partial.tsv')
HEADER = chr(9).join(['schema_' + SCHEMA, 'pos', 'aa'] + cols)
done = set()
if os.path.exists(CKPT) and os.path.getsize(CKPT) > 0:
    first = open(CKPT).readline().rstrip(chr(10))
    if first != HEADER:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; '
                         'delete it and rerun' % CKPT)
    for line in open(CKPT):
        p = line.split(chr(9))
        if len(p) > 2 and p[1].isdigit():
            done.add((int(p[1]), p[2]))
    log('resuming, already scored %d variants' % len(done))

todo = [v for v in variants if v not in done]

import torch
t0 = time.time()
with open(CKPT, 'a') as fh:
    if not done:
        fh.write(HEADER + chr(10))
        fh.flush()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        batch = base_ids.repeat(len(chunk), 1)
        for k, (j, y) in enumerate(chunk):
            batch[k, j - 1 + offset] = tok.convert_tokens_to_ids(y)
        with torch.no_grad():
            logits = model(batch.cuda()).logits
        sel = torch.softmax(logits[:, read_idx, :].float(), dim=-1)
        for k, (j, y) in enumerate(chunk):
            ps = ['%.6e' % float(sel[k, m, read_tok[m]]) for m in range(len(read_idx))]
            fh.write(chr(9).join(['row', str(j), y] + ps) + chr(10))
        fh.flush()
        if (b // BATCH) % 25 == 0:
            log('scored %d of %d, %.1f min elapsed'
                % (b + len(chunk), len(todo), (time.time() - t0) / 60))

log('scan complete, %.1f min' % ((time.time() - t0) / 60))


In [ ]:
scan = [l.rstrip(chr(10)).split(chr(9)) for l in open(CKPT)]
body = [r for r in scan[1:] if len(r) > 2 and r[1].isdigit()]
if len(body) != len(variants):
    raise SystemExit('FATAL scan has %d rows, expected %d' % (len(body), len(variants)))
cidx = {c: i + 3 for i, c in enumerate(cols)}

rescue = os.path.join(OUT, 'rescue_scan.tsv')
n_cross = 0
with open(rescue, 'w') as fh:
    fh.write(chr(9).join(['rescuer_pos', 'rescuer_from', 'rescuer_to', 'target_pos',
                          'target_human_aa', 'p_target', 'p_baseline', 'fold_change',
                          'crosses_threshold', 'separation',
                          'target_identical_to_human',
                          'rescuer_identical_to_human']) + chr(10))
    for r in body:
        j, y = int(r[1]), r[2]
        hj = CK2HU.get(j)
        rid = hj is not None and chicken[j - 1] == human[hj - 1]
        for t in targets:
            p = float(r[cidx['p_%d' % t['chicken_pos']]])
            cross = p >= THRESHOLD
            n_cross += cross
            fh.write(chr(9).join([
                str(j), chicken[j - 1], y, str(t['chicken_pos']), t['human_aa'],
                '%.6e' % p, '%.6e' % t['p_baseline'],
                '%.3f' % (p / t['p_baseline']), str(cross),
                str(abs(j - t['chicken_pos'])), 'False', str(rid)]) + chr(10))

gate_out = os.path.join(OUT, 'gate_perturbation.tsv')
with open(gate_out, 'w') as fh:
    fh.write(chr(9).join(['variant_pos', 'variant_from', 'variant_to', 'gate_pos',
                          'gate_aa', 'p_gate_wt', 'separation']) + chr(10))
    for r in body:
        j, y = int(r[1]), r[2]
        for g in gates:
            fh.write(chr(9).join([
                str(j), chicken[j - 1], y, str(g['chicken_pos']), g['chicken_aa'],
                r[cidx['g_%d' % g['chicken_pos']]],
                str(abs(j - g['chicken_pos']))]) + chr(10))

meta = {'model': MODEL, 'schema': SCHEMA, 'threshold': THRESHOLD,
        'n_variants': len(variants), 'n_targets': len(targets), 'n_gates': len(gates),
        'n_crossing_threshold': n_cross,
        'targets': targets, 'gates': gates,
        'chicken_sha256': hashlib.sha256(chicken.encode()).hexdigest(),
        'finished_utc': datetime.datetime.utcnow().isoformat(timespec='seconds')}
with open(os.path.join(OUT, 'run_meta.json'), 'w') as fh:
    json.dump(meta, fh, indent=2)

for f in (rescue, gate_out):
    if not os.path.exists(f) or os.path.getsize(f) < 1000:
        raise SystemExit('FATAL %s missing or too small' % f)
log('wrote rescue_scan.tsv (%d bytes) and gate_perturbation.tsv (%d bytes); '
    'pairs crossing the threshold %d of %d'
    % (os.path.getsize(rescue), os.path.getsize(gate_out),
       n_cross, len(variants) * len(targets)))
